In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "260_build_silver_fact_bid_item.py"
RUN_ID = str(uuid.uuid4())

VENDOR_LATEST_TABLE = f"{catalog}.silver.closed_bids_vendor_item_latest"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_VENDOR_TABLE = f"{catalog}.silver.dim_vendor"
DIM_PROJECT_CLASSIFICATION_TABLE = f"{catalog}.silver.dim_project_classification"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
TARGET_TABLE = f"{catalog}.silver.fact_bid_item"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build fact_bid_item
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH src AS (
        SELECT
              md5(COALESCE(v.base_row_key, '')) AS bid_item_fact_key
            , dp.project_key
            , dv.vendor_key
            , v.item_specification_key

            , pc.project_classification_key
            , dc.county_key

            , v.base_row_key
            , v.source_row_id
            , v.source_created_at
            , v.source_updated_at
            , v.source_version
            , v.ingestion_run_id
            , v.ingested_at_utc

            , v.source_name

            , v.project_id
            , v.vendor_name
            , v.vendor_name_standardized

            , v.bid_submit_sequence_number
            , v.bid_rank_sequence_number
            , v.low_bidder_flag
            , CASE
                  WHEN v.low_bidder_flag = TRUE THEN 1
                  ELSE 0
              END AS low_bidder_flag_int
            , v.dbe_goal_percent
            , v.maximum_number_of_working

            , v.bid_code
            , v.alternative_bid_code
            , v.bid_item_sequence_number
            , v.bid_item_description
            , v.measurement_unit
            , v.bid_item_quantity
            , v.bid_item_unit_price_amount
            , v.bid_total_amount

            , v.specification_code
            , v.specification_description
            , v.effective_specification_description
            , v.spec_book_year

            , v.force_account_amount
            , v.force_account_description
            , v.nbi_number
            , v.utility_id

            , v.business_submission_key
            , v.business_item_key
            , v.record_hash

            , v.item_version_count
            , v.min_submit_sequence_for_item
            , v.max_submit_sequence_for_item
            , v.vendor_project_submit_sequence_count
            , v.latest_submit_for_vendor_project

            , v.vendor_project_has_resubmission_history
            , v.item_has_resubmission_history
            , v.item_changed_across_submit_sequences
            , v.is_latest_submit_for_vendor_project
            , v.item_missing_from_latest_project_submit
            , v.item_changed_value_across_submits

            , CASE
                  WHEN v.item_specification_key IS NOT NULL THEN TRUE
                  ELSE FALSE
              END AS is_standard_specification_item

            , CASE
                  WHEN v.item_specification_key IS NULL THEN TRUE
                  ELSE FALSE
              END AS is_unmapped_specification_item

            , CASE
                  WHEN COALESCE(v.bid_code, '') LIKE '9602-%'
                    OR UPPER(COALESCE(v.bid_item_description, '')) LIKE '%PAYMENT ADJUSTMENT%'
                  THEN TRUE
                  ELSE FALSE
              END AS is_payment_adjustment_item

            , CASE
                  WHEN COALESCE(v.bid_code, '') LIKE '9606-%'
                    OR UPPER(COALESCE(v.bid_item_description, '')) LIKE '%SPECIAL DEDUCTION%'
                  THEN TRUE
                  ELSE FALSE
              END AS is_special_deduction_item

            , CASE
                  WHEN v.item_specification_key IS NULL
                   AND (
                        COALESCE(v.bid_code, '') LIKE '9602-%'
                        OR UPPER(COALESCE(v.bid_item_description, '')) LIKE '%PAYMENT ADJUSTMENT%'
                        OR COALESCE(v.bid_code, '') LIKE '9606-%'
                        OR UPPER(COALESCE(v.bid_item_description, '')) LIKE '%SPECIAL DEDUCTION%'
                   )
                  THEN TRUE
                  ELSE FALSE
              END AS is_nonstandard_adjustment_item

            , CASE
                  WHEN v.bid_item_quantity IS NOT NULL
                   AND v.bid_item_unit_price_amount IS NOT NULL
                  THEN v.bid_item_quantity * v.bid_item_unit_price_amount
                  ELSE NULL
              END AS extended_item_amount_calc

            , CASE
                  WHEN v.bid_item_quantity IS NOT NULL
                   AND v.bid_item_unit_price_amount IS NOT NULL
                   AND v.bid_total_amount IS NOT NULL
                  THEN ABS(
                       v.bid_total_amount
                       - (v.bid_item_quantity * v.bid_item_unit_price_amount)
                  )
                  ELSE NULL
              END AS project_total_vs_extended_item_abs_diff

        FROM {VENDOR_LATEST_TABLE} v

        LEFT JOIN {DIM_PROJECT_TABLE} dp
            ON v.project_id = dp.project_id

        LEFT JOIN {DIM_VENDOR_TABLE} dv
            ON v.vendor_name_standardized = dv.vendor_name_standardized

        LEFT JOIN {DIM_PROJECT_CLASSIFICATION_TABLE} pc
            ON (
                CASE
                    WHEN COALESCE(v.project_classification, '') = '' THEN 'UNKNOWN'
                    ELSE v.project_classification
                END
            ) = pc.project_classification

        LEFT JOIN {DIM_COUNTY_TABLE} dc
            ON (
                CASE
                    WHEN COALESCE(v.county, '') = '' THEN 'UNKNOWN'
                    WHEN v.county = 'De Witt' THEN 'DeWitt'
                    ELSE v.county
                END
            ) = dc.county
    )

    SELECT
          bid_item_fact_key
        , project_key
        , vendor_key
        , item_specification_key

        , project_classification_key
        , county_key

        , base_row_key
        , source_row_id
        , source_created_at
        , source_updated_at
        , source_version
        , ingestion_run_id
        , ingested_at_utc

        , source_name

        , project_id
        , vendor_name
        , vendor_name_standardized

        , bid_submit_sequence_number
        , bid_rank_sequence_number
        , low_bidder_flag
        , low_bidder_flag_int
        , dbe_goal_percent
        , maximum_number_of_working

        , bid_code
        , alternative_bid_code
        , bid_item_sequence_number
        , bid_item_description
        , measurement_unit
        , bid_item_quantity
        , bid_item_unit_price_amount
        , bid_total_amount

        , specification_code
        , specification_description
        , effective_specification_description
        , spec_book_year

        , force_account_amount
        , force_account_description
        , nbi_number
        , utility_id

        , business_submission_key
        , business_item_key
        , record_hash

        , item_version_count
        , min_submit_sequence_for_item
        , max_submit_sequence_for_item
        , vendor_project_submit_sequence_count
        , latest_submit_for_vendor_project

        , vendor_project_has_resubmission_history
        , item_has_resubmission_history
        , item_changed_across_submit_sequences
        , is_latest_submit_for_vendor_project
        , item_missing_from_latest_project_submit
        , item_changed_value_across_submits

        , is_standard_specification_item
        , is_unmapped_specification_item
        , is_payment_adjustment_item
        , is_special_deduction_item
        , is_nonstandard_adjustment_item

        , extended_item_amount_calc
        , project_total_vs_extended_item_abs_diff
    FROM src
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Item-level bid fact table built from silver.closed_bids_vendor_item_latest and linked to project, vendor, classification, and county dimensions.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    missing_project_classification_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE project_classification_key IS NULL
    """).collect()[0]["missing_count"]

    missing_county_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE county_key IS NULL
    """).collect()[0]["missing_count"]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)
    print("Validation:")
    print(f"  missing_project_classification_keys: {missing_project_classification_keys:,}")
    print(f"  missing_county_keys: {missing_county_keys:,}")

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise